# protein

> ProtST under the sysone head: its two towers loaded from tensors alone, zero-shot typed answers from its joint space, and the protein application, where the protein is a side stream beside a text row.

In [ ]:
#| default_exp protein

In [ ]:
#| export
import json
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, BertConfig, BertModel, EsmConfig, EsmModel

from sysone.core import *
from sysone.template import Stream, TokenStream
from sysone.data import TypedDecisions
from sysone.models import *
from sysone.zeroshot import Embedder, SimilarityDecider
from sysone.learner import Learner
from sysone.learner import decision_learner as _decision_learner
from sysone.cache import FeatureCache        # and the verbs the cache adds to Learner

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import logging, tempfile
import torch.nn.functional as F
from safetensors.torch import load_file
from sysone.evaluate import evaluate
from sysone.inference import Decider
from sysone.template import show_row
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()

This notebook adds a protein encoder the way the [multimodal](20_neomme.ipynb) notebook added an image encoder: load it and look at it, check it against its reference, write the spec, run one forward pass, train a head, and write down what to watch.

[ProtST](https://arxiv.org/abs/2301.12040) (Xu et al., ICML 2023) has two towers: ESM-1b, a protein language model (33 layers, width 1,280, 650M parameters), and PubMedBERT, a biomedical text encoder (12 layers, width 768). They were trained together on ProtDescribe, Swiss-Prot sequences paired with text about each protein's function, location, family and name. Among the training objectives is a CLIP-style alignment. It pools each tower into one vector, projects both into a 512-wide space, and pulls a protein and its own description together there. That joint space is what sysone uses, in two ways:

- **Zero-shot.** Embed the protein and each option's text in the joint space. An option's logit is ProtST's logit scale times the cosine between the two. This is the `SimilarityDecider` of [zeroshot](11_zeroshot.ipynb), and it needs no training at all.
- **A decision model.** The protein is a *side stream* ([template](01_template.ipynb#side-stream-inputs)): ESM reads it and pools it to one vector. PubMedBERT reads a text row as for any sysone encoder (the question, and an anchor before each option). A pair head scores each option's anchor against the protein's vector. The towers don't talk to each other (late fusion), so both can stay frozen and cached, and only the head trains. [Cross-attention](03_models.ipynb#cross-attention-from-a-side-stream) is the other design, where the text attends to the protein inside the encoder.

```{mermaid}
flowchart LR
    P["a protein sequence"] --> E["ESM-1b: the mean over<br/>residues, projected"]
    Q["a typed question:<br/>its options' texts"] --> B["PubMedBERT: the mean<br/>over tokens, projected"]
    E --> Z["zero-shot: scale × cosine<br/>per option"]
    B --> Z
    Q --> R["a text row: the question,<br/>an anchor per option"]
    R --> B2["PubMedBERT:<br/>a vector per anchor"]
    E --> H["pair head: each anchor<br/>against the protein"]
    B2 --> H
    Z --> J["÷ temperature, softmax:<br/>the Jev answer"]
    H --> J
```

## 1. Load and inspect

The Hub port of ProtST, [MilaDeepGraph/ProtST-ESM1b](https://huggingface.co/MilaDeepGraph/ProtST-ESM1b), keeps the two towers and their projection heads, and not the fusion module used for multimodal mask prediction. Its tensors sit under `protein_model.esm` and `text_model.bert`. Each tower has two heads (dense → ReLU → out_proj, into the 512-wide space): `protein_mlp` and `text_mlp` for the pooled vector, and `residue_mlp` and `word_mlp` for each token's vector. A scalar `logit_scale` completes the checkpoint. The repo ships its own modeling code, which transformers runs only with `trust_remote_code=True`.

sysone doesn't run that code. The towers are transformers' own `EsmModel` and `BertModel`, so sysone builds them from the two configs inside `config.json` and reads only the tensors. It uses `model.safetensors` when the repo has one, and otherwise `pytorch_model.bin` through `torch.load(weights_only=True)`, which unpickles tensors and refuses anything else. The file is memory-mapped, so the 1.5 GB bf16 checkpoint is read as the towers are filled, not held in memory beside them. The towers are built straight from those tensors, not initialised at random first, and are float32 unless asked otherwise.

The port has no tokenizer files. ProtST was trained with ESM-1b's and PubMedBERT's, and those are the defaults. A folder with `protein_tokenizer/` and `text_tokenizer/` inside, as the fixture has, brings its own.

In [ ]:
#| export
ENCODER = "MilaDeepGraph/ProtST-ESM1b"
TOKENIZERS = {"protein": "facebook/esm1b_t33_650M_UR50S", "text": "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"}

def _file(id:str, name:str, revision:str|None=None) -> Path:
    "A file of a local folder, or of a Hub repo (downloaded once, then read from the cache)"
    if Path(id).exists():
        if not (Path(id) / name).exists(): raise FileNotFoundError(Path(id) / name)
        return Path(id) / name
    from huggingface_hub import hf_hub_download
    return Path(hf_hub_download(id, name, revision=revision))

def is_protst(id:str, revision:str|None=None) -> bool:
    "Whether `id` (a folder or a Hub repo) holds a ProtST checkpoint: a config whose model_type is 'protst'"
    try: return json.loads(_file(id, "config.json", revision).read_text()).get("model_type") == "protst"
    except Exception: return False

def protst_config(id:str=ENCODER, revision:str|None=None) -> SimpleNamespace:
    "The checkpoint's configuration: its model type, and the ESM and BERT configs of its towers"
    cfg = json.loads(_file(id, "config.json", revision).read_text())
    return SimpleNamespace(model_type="protst", protein_config=EsmConfig(**cfg["protein_config"]), text_config=BertConfig(**cfg["text_config"]))

def protst_tensors(id:str=ENCODER, revision:str|None=None) -> dict:
    "The checkpoint's tensors: `model.safetensors` if there is one, else `pytorch_model.bin` read with `weights_only=True` (tensors, no code), memory-mapped"
    try: f = _file(id, "model.safetensors", revision)
    except Exception: f = _file(id, "pytorch_model.bin", revision)
    if f.suffix == ".safetensors":
        from safetensors.torch import load_file
        return load_file(str(f))
    return torch.load(f, map_location="cpu", weights_only=True, mmap=True)

def protst_tokenizers(id:str=ENCODER, protein:str|None=None, text:str|None=None) -> tuple:
    "The protein and the text tokenizer: the folder's own (`protein_tokenizer/`, `text_tokenizer/`), else ESM-1b's and PubMedBERT's"
    def pick(kind, given):
        if given is not None: return given
        own = Path(id) / f"{kind}_tokenizer"
        return str(own) if own.exists() else TOKENIZERS[kind]
    return AutoTokenizer.from_pretrained(pick("protein", protein)), AutoTokenizer.from_pretrained(pick("text", text))

A tower is built by transformers' `from_pretrained` from its config and its tensors, with no pooling layer (ProtST pools by a mean, below). The build checks that every weight the tower needs is in the checkpoint and that nothing is left over. A projection head becomes a `ProjMLP`, whose Linear → ReLU → Linear is exactly ProtST's dense → ReLU → out_proj.

In [ ]:
#| export
def _tower(cls, config, sd:dict, prefix:str, dtype):
    "A transformers tower built from its config and the tensors under `prefix`; nothing is initialised at random"
    from transformers.utils import logging as hf_logging
    part = {k[len(prefix):]: v for k, v in sd.items() if k.startswith(prefix)}
    bar = hf_logging.is_progress_bar_enabled(); hf_logging.disable_progress_bar()
    try: m, info = cls.from_pretrained(None, config=config, state_dict=part, add_pooling_layer=False, dtype=dtype, output_loading_info=True)
    finally:
        if bar: hf_logging.enable_progress_bar()
    if info["missing_keys"] or info["unexpected_keys"]:
        raise ValueError(f"{prefix}: the checkpoint misses {sorted(info['missing_keys'])[:3]} and has unused {sorted(info['unexpected_keys'])[:3]}")
    return m.eval()

def _proj(sd:dict, prefix:str, dtype) -> ProjMLP:
    "A ProtST projection head (dense → ReLU → out_proj) as a `ProjMLP`"
    w1, w2 = sd[prefix + "dense.weight"], sd[prefix + "out_proj.weight"]
    m = ProjMLP([w1.shape[1], w1.shape[0], w2.shape[0]])
    m.load_state_dict({"0.weight": w1, "0.bias": sd[prefix + "dense.bias"], "2.weight": w2, "2.bias": sd[prefix + "out_proj.bias"]})
    return m.to(dtype).eval()

class ProtST(nn.Module):
    "ProtST's two towers and the heads that project their pooled vectors into one space: ESM over proteins, BERT over text"
    def __init__(self, esm:nn.Module, bert:nn.Module, protein_proj:ProjMLP, text_proj:ProjMLP, logit_scale:float,
                 protein_tokenizer, text_tokenizer, config=None, id:str|None=None, revision:str|None=None):
        super().__init__()
        self.esm, self.bert, self.protein_proj, self.text_proj = esm, bert, protein_proj, text_proj
        self.logit_scale, self.protein_tokenizer, self.text_tokenizer = logit_scale, protein_tokenizer, text_tokenizer
        self.config, self.id, self.revision = config, id, revision

    def __repr__(self):
        pc, tc = self.esm.config, self.bert.config
        n = sum(p.numel() for p in self.parameters())
        return (f"ProtST({self.id} · protein: ESM {pc.num_hidden_layers} × {pc.hidden_size} · text: BERT {tc.num_hidden_layers} × "
                f"{tc.hidden_size} · joint space {self.protein_proj.dims[-1]} · scale {self.logit_scale:.3g} · {n / 1e6:,.1f}M params · "
                f"{next(self.parameters()).dtype} on {self.device})")

    @property
    def device(self) -> torch.device: return next(self.esm.parameters()).device

def load_protst(id:str=ENCODER, revision:str|None=None, dtype=torch.float32, device=None,
                protein_tokenizer:str|None=None, text_tokenizer:str|None=None) -> ProtST:
    "ProtST from a Hub repo or a folder: transformers' ESM and BERT filled with the checkpoint's tensors (its modeling code is not run)"
    config, sd = protst_config(id, revision), protst_tensors(id, revision)
    esm = _tower(EsmModel, config.protein_config, sd, "protein_model.esm.", dtype)
    bert = _tower(BertModel, config.text_config, sd, "text_model.bert.", dtype)
    pp, tp = _proj(sd, "protein_model.protein_mlp.", dtype), _proj(sd, "text_model.text_mlp.", dtype)
    scale = float(sd["logit_scale"].float().exp())
    del sd
    p = ProtST(esm, bert, pp, tp, scale, *protst_tokenizers(id, protein_tokenizer, text_tokenizer), config=config, id=str(id), revision=revision)
    return p.to(device) if device is not None else p

The fixture is a random two-tower model laid out as the Hub port is: a 2-layer, 32-wide ESM and BERT, heads into a 16-wide space, and ESM's own 33-token protein tokenizer beside a small WordPiece text tokenizer (`python nbs/fixtures/make_fixtures.py protst` rebuilds it offline). It loads the way the real checkpoint does:

In [ ]:
FIX = "fixtures/tiny-protst"
p = load_protst(FIX)
p

In [ ]:
test_eq(is_protst(FIX), True); test_eq(is_protst("fixtures/tiny-modernbert"), False)
test_close(p.logit_scale, float(np.exp(2.6592)), eps=1e-3)
test_eq((p.protein_proj.dims, p.text_proj.dims), ([32, 32, 16], [32, 32, 16]))
test_eq((p.protein_tokenizer.cls_token_id, p.protein_tokenizer.pad_token_id, p.protein_tokenizer.eos_token_id), (0, 1, 2))
test_eq(p.protein_tokenizer("MKTAY")["input_ids"], [0] + p.protein_tokenizer("M K T A Y", add_special_tokens=False)["input_ids"] + [2])

The real checkpoint, loaded the same way on an M3 Pro, gives `ProtST(MilaDeepGraph/ProtST-ESM1b · protein: ESM 33 × 1280 · text: BERT 12 × 768 · joint space 512 · scale 14.2 · 762.9M params · torch.float32 on mps:0)`. Every weight the two towers need is in the checkpoint and none is left over, down to ESM's contact head, which sysone never calls. Once the file is in the cache, it loads in one to two seconds and holds 3.0 GB of float32 weights. The logit scale is 2.656, which is $\ln(1/0.07)$ rounded to bf16. That is where CLIP-style training starts the scale, so the port's zero-shot logits are the cosine times 14.2.

## 2. Zero-shot

ProtST pools a protein as the mean of ESM's last hidden states over its residues, leaving out `<cls>`, `<eos>` and padding, and projects the mean with `protein_mlp`. A text is pooled the same way over its tokens and projected with `text_mlp`. The port's zero-shot example feeds a label's text as `[CLS]` followed by its tokens, without a `[SEP]`, and sysone does the same. Both vectors are normalised, and an option's logit is the scale times their cosine.

`SideEncoder` already pools that way, so the two embedders are side encoders over ProtST's towers and heads, and the decision model below reuses the same code. The protein embedder builds its input the way the `protein` stream does, cropping at the ends (see [the spec](#the-spec)). Zero-shot and the decision model therefore read the same residues. It also batches proteins by length, so a batch is mostly residues rather than padding.

In [ ]:
#| export
def clean_sequence(s) -> str:
    "A protein sequence as ESM's tokenizer reads it: one-letter codes in capitals, no spaces"
    return "".join(str(s).split()).upper()

def protein_specials(tok) -> tuple:
    "The ids a protein's mean leaves out: its tokenizer's cls, eos and pad"
    return tuple(i for i in (tok.cls_token_id, tok.eos_token_id, tok.pad_token_id) if i is not None)

@torch.no_grad()
def _pooled(side:SideEncoder, ids:list, pad:int, batch_size:int, device) -> np.ndarray:
    "A side encoder's pooled vectors for lists of token ids, batched by length, in eval mode, in the input order"
    modes = [(m, m.training) for m in side.modules()]       # the towers may be shared: each module gets its own mode back
    side.eval()
    out, order = [None] * len(ids), sorted(range(len(ids)), key=lambda i: len(ids[i]))
    try:
        for i in range(0, len(order), batch_size):
            chunk = order[i:i + batch_size]; L = max(len(ids[j]) for j in chunk)
            x = torch.full((len(chunk), L), pad, dtype=torch.long); m = torch.zeros_like(x)
            for r, j in enumerate(chunk): x[r, :len(ids[j])] = torch.tensor(ids[j]); m[r, :len(ids[j])] = 1
            v = side(x.to(device), m.to(device))[1].float().cpu().numpy()
            for r, j in enumerate(chunk): out[j] = v[r]
    finally:
        for m, t in modes: m.training = t
    return np.stack(out)

class ProteinEmbedder(Embedder):
    "Protein sequences to ProtST's protein embeddings: ESM over the residues (cropped as the `protein` stream crops them), their mean, projected"
    def __init__(self, protst:ProtST, max_len:int=1024, crop:str="ends", batch_size:int=8):
        self.protst, self.batch_size = protst, batch_size
        self.stream = TokenStream("protein", protst.protein_tokenizer, max_len=max_len, crop=crop)
        self.side = SideEncoder(protst.esm, protein_specials(protst.protein_tokenizer), proj=protst.protein_proj)
    def __repr__(self): return f"ProteinEmbedder({self.stream.max_len} tokens, crop {self.stream.crop!r}, batches of {self.batch_size})"
    def embed(self, items):
        ids = [self.stream.build(clean_sequence(s))[self.stream.columns[0]] for s in items]
        return _pooled(self.side, ids, self.protst.protein_tokenizer.pad_token_id, self.batch_size, self.protst.device)

class TextEmbedder(Embedder):
    "Texts to ProtST's text embeddings, as its zero-shot example makes them: [CLS] and the text's tokens (no [SEP]), their mean, projected"
    def __init__(self, protst:ProtST, max_len:int=128, batch_size:int=32):
        self.protst, self.max_len, self.batch_size = protst, max_len, batch_size
        tok = protst.text_tokenizer
        self.side = SideEncoder(protst.bert, (tok.cls_token_id, tok.sep_token_id, tok.pad_token_id), proj=protst.text_proj)
    def embed(self, items):
        tok = self.protst.text_tokenizer
        ids = [[tok.cls_token_id] + tok(str(t), add_special_tokens=False, truncation=True, max_length=self.max_len)["input_ids"] for t in items]
        return _pooled(self.side, ids, tok.pad_token_id or 0, self.batch_size, self.protst.device)

def zero_shot(protst:ProtST, prompt:str="{text}", proteins:Embedder|None=None, texts:Embedder|None=None, state_key:str="sequence",
              max_len:int=1024, crop:str="ends", **kw) -> SimilarityDecider:
    "A zero-shot decider on ProtST: an option's logit is ProtST's scale times the cosine between the protein's embedding and the option text's"
    return SimilarityDecider(proteins or ProteinEmbedder(protst, max_len, crop), texts or TextEmbedder(protst), scale=protst.logit_scale,
                             prompt=prompt, state_key=state_key, **({"batch_size": 64, "name": "protst"} | kw))

To check the port's protocol, here it is written from its modeling code: the tower, the mean over the non-special tokens, then the head's two linear maps read straight from the checkpoint's tensors. The embedders give the same unit vectors:

In [ ]:
sd = load_file(f"{FIX}/model.safetensors")

def hub_feature(model, head, ids, specials):
    "A pooled vector as the Hub port computes it: its tower, the mean over the non-special tokens, then its head (dense → ReLU → out_proj)"
    h = model(input_ids=ids, attention_mask=torch.ones_like(ids)).last_hidden_state
    keep = ~torch.isin(ids, torch.tensor(specials))
    pooled = (h * keep[..., None]).sum(1) / (keep.sum(1, keepdim=True) + 1e-6)
    x = torch.relu(pooled @ sd[head + "dense.weight"].T + sd[head + "dense.bias"])
    return F.normalize(x @ sd[head + "out_proj.weight"].T + sd[head + "out_proj.bias"], dim=-1)

seqs = ["MKTAYIAKQRQISFVKSHFSRQ", "MSDNGPQNQRNAPRITFGGPSDSTGSNQNGERSGARSKQRRPQGLPNNTASWFTALTQHGK"]
texts = ["SUBCELLULAR LOCATION: Nucleus.", "a membrane protein"]
with torch.no_grad():
    ref_p = torch.cat([hub_feature(p.esm, "protein_model.protein_mlp.", p.protein_tokenizer(s, return_tensors="pt")["input_ids"], (0, 1, 2)) for s in seqs])
    tt = p.text_tokenizer
    ref_t = torch.cat([hub_feature(p.bert, "text_model.text_mlp.", torch.tensor([[tt.cls_token_id] + tt(t, add_special_tokens=False)["input_ids"]]),
                                   (tt.cls_token_id, tt.sep_token_id, tt.pad_token_id)) for t in texts])
test_close(ProteinEmbedder(p)(seqs), ref_p.numpy(), eps=1e-5)
test_close(TextEmbedder(p)(texts), ref_t.numpy(), eps=1e-5)
test_close(ProteinEmbedder(p)([" ".join(seqs[0]).lower()]), ref_p[:1].numpy(), eps=1e-5)     # spaced or lower-case, the same protein
test_eq((p.esm.training, p.bert.training), (False, False))                                    # the towers keep their mode

::: {.callout-note title="Finding: an embedder switched the shared text tower into training mode"}
The first `_pooled` put its side encoder in eval mode for the pass and restored the encoder's own mode afterwards. A module built on the spot starts in training mode, though, so "restoring" it switched the towers it wraps, which are ProtST's shared towers, into training mode. PubMedBERT has dropout (0.1) and ESM has none, so the protein vectors stayed right while every later text embedding came out noisy. It showed up as zero-shot logits that disagreed with the Hub port's protocol computed by hand, but only when the hand computation ran after an embedder. `_pooled` now gives every module back its own mode, and the test above checks that both towers are still in eval mode.
:::

`zero_shot` puts the two embedders under a `SimilarityDecider`. The state holds the protein under `sequence`, and each question's options become texts through the prompt. A choice's options become their descriptions (or their keys), and a noul's become its two sides, `criteria={"false": …, "true": …}`. The fixture's towers are random, so the answers below are meaningless. What matters is their shape: the Jev answers a trained sysone model would give.

In [ ]:
LOCS = {"Nucleus": "SUBCELLULAR LOCATION: Nucleus.", "Cytoplasm": "SUBCELLULAR LOCATION: Cytoplasm.",
        "Cell membrane": "SUBCELLULAR LOCATION: Cell membrane.", "Extracellular": "SUBCELLULAR LOCATION: Secreted."}
QS = {"location": {"type": "choice", "instructions": "Where in the cell is this protein?", "criteria": LOCS},
      "membrane": {"type": "noul", "instructions": "The protein is bound to a membrane.",
                   "criteria": {"false": "a soluble protein", "true": "a membrane protein"}}}
zs = zero_shot(p)
ans = zs.predict({"sequence": seqs[0]}, QS)
test_eq((sorted(ans), ans["location"]["choice"] in LOCS, 0 <= ans["membrane"]["noul"] <= 1), (["location", "membrane"], True, True))
cos = ref_p[0] @ torch.cat([hub_feature(p.bert, "text_model.text_mlp.", torch.tensor([[tt.cls_token_id] + tt(t, add_special_tokens=False)["input_ids"]]),
                                        (tt.cls_token_id, tt.sep_token_id, tt.pad_token_id)) for t in LOCS.values()]).T
test_close(zs.row_logits([{"state": {"sequence": seqs[0]}}], [Question.from_dict(QS["location"], name="location")])[0], p.logit_scale * cos.detach().numpy(), eps=1e-4)
zs, ans

## 3. The spec

The decision model reads two inputs. PubMedBERT reads the text row, built by the `auto` template (Laya's layout on the tokenizer's own `[CLS]`, `[SEP]` and `[MASK]`): the question, and an anchor before each option. ESM reads the `protein` stream, a `TokenStream` on ESM's tokenizer. ESM-1b has 1,026 positions and is trained on 1,024-token crops, so `protein_len` is at most 1,024 tokens, the two special tokens included. A longer protein is cropped at the `ends`: the stream keeps its start and its end and drops the middle. That's where most of what a localization question turns on sits: N-terminal signal peptides and transit peptides, and C-terminal signals such as the peroxisomal SKL and the ER's KDEL.

`protst_spec` builds the spec from a Hub id, a folder, or a `ProtST` already loaded. The spec is cheap, since it reads only the config and the two tokenizers, and its loader builds the encoder when a learner asks for it. A spec made from a loaded `ProtST` hands out that model's own towers without loading them again, so a notebook can run zero-shot and train a head over one copy of the weights. Training the towers (LoRA, `full`) then changes that copy too, so for those regimes load a fresh one.

`sysone.protein` registers ProtST as an encoder source, so `EncoderSpec.from_pretrained` recognises a ProtST checkpoint by its config, and an export that names the source loads it back. The family's LoRA targets are the text tower's attention projections, and the protein tower stays frozen. PubMedBERT is the small tower, and it's the one reading the question.

In [ ]:
#| export
FAMILIES["protst"] = ("auto", r"text\.encoder\.layer\.\d+\.attention\.(self\.(query|key|value)|output\.dense)")

def stream_encoder(protst:ProtST) -> StreamEncoder:
    "ProtST as a stream encoder: BERT reads the row, and ESM reads the `protein` stream, pooled and projected as for zero-shot"
    return StreamEncoder(protst.bert, {"protein": SideEncoder(protst.esm, protein_specials(protst.protein_tokenizer), proj=protst.protein_proj)})

def protst_spec(protst:ProtST|str=ENCODER, revision:str|None=None, protein_len:int=1024, crop:str="ends",
                protein_tokenizer:str|None=None, text_tokenizer:str|None=None, **kw) -> EncoderSpec:
    "The spec of ProtST: its text tokenizer for the row, a `protein` stream on its protein tokenizer, and the stream encoder, loaded when asked for"
    if isinstance(protst, ProtST):
        bundle, id, revision, config = protst, protst.id, protst.revision, protst.config
        ptok, ttok = protst.protein_tokenizer, protst.text_tokenizer
        loader = lambda spec, dtype=None, **kw: stream_encoder(bundle.to(dtype=dtype) if dtype is not None else bundle)
    else:
        id, config = str(protst), protst_config(str(protst), revision)
        ptok, ttok = protst_tokenizers(id, protein_tokenizer, text_tokenizer)
        loader = lambda spec, dtype=torch.float32, **kw: stream_encoder(load_protst(id, revision, dtype=dtype or torch.float32,
                                                                                    protein_tokenizer=protein_tokenizer, text_tokenizer=text_tokenizer))
    return EncoderSpec(id, config, ttok, revision=revision, source="protst", loader=loader,
                       streams=[TokenStream("protein", ptok, max_len=protein_len, crop=crop)])

ENCODER_SOURCES.append(("protst", is_protst, protst_spec))

In [ ]:
spec = protst_spec(p)
test_eq((spec.family, spec.width, spec.context, spec.source, list(spec.streams)), ("protst", 32, 512, "protst", ["protein"]))
test_eq(EncoderSpec.from_pretrained(FIX).source, "protst")             # recognised by its config
test_fail(lambda: protst_spec(p, protein_len=2), contains="no room")
spec

A record keeps the protein in its state. `Stream("sequence", "protein")` moves it out of the text and into the stream, and the protein application adds that transform when the data has none. Each row of a record carries the same protein ids, so several questions about one protein share one input:

In [ ]:
rec = {"id": "p0", "state": {"sequence": seqs[1]}, "questions": QS, "gold": {"location": "Nucleus", "membrane": False}}
rows = spec.builder(tfms=[Stream("sequence", "protein")]).build_record(rec)
print(show_row(rows[0], spec.tokenizer, max_chars=400))
test_eq([len(r.extras["protein_input_ids"]) for r in rows], [len(seqs[1]) + 2] * 2)
test_eq(rows[0].extras["protein_input_ids"][:3], [0] + p.protein_tokenizer("MS", add_special_tokens=False)["input_ids"])

## 4. One forward pass

The stream encoder gives the text's hidden states and the protein's pooled vector, which the head reads as its query (`query="stream:protein"`). The protein vector is ProtST's own protein embedding before it's normalised. So the head starts from the representation zero-shot uses, and a cosine of 1 against the zero-shot embedder confirms that:

In [ ]:
b = collate(rows, spec.tokenizer.pad_token_id)
se = spec.load_encoder()
model = EncoderDecisionModel(se, DecisionHead(spec.width, layers=0, scorer="pair", query="stream:protein", query_width=16), spec).eval()
enc_in = {k: b[k] for k in ("input_ids", "attention_mask", "protein_input_ids", "protein_attention_mask")}
with torch.no_grad():
    out = se(**enc_in)
    logits = model(**{k: v for k, v in b.items() if k not in ("target", "label")})
test_eq((out.last_hidden_state.shape[0], out.stream_vectors["protein"].shape, logits.shape), (2, (2, 16), (2, 4)))
test_close(F.cosine_similarity(out.stream_vectors["protein"][:1], torch.tensor(ProteinEmbedder(p)([seqs[1]]))).item(), 1.0, eps=1e-5)
test_eq(se.text is p.bert and se.sides["protein"].model is p.esm, True)          # the loaded ProtST's own towers
logits

## 5. A head-only run

`decision_learner` is the protein application. The encoder is a Hub id, a folder, a loaded `ProtST` or its spec. By default it adds the `protein` stream, freezes both towers, caches each row's anchors and protein vector, and trains a pair head on the standardised vectors. The cache is float32. Two questions about one protein go through ESM once. The cache first encodes the distinct proteins in order of length, under `stream_memo`, so the rows' own batches find every protein already encoded.

In [ ]:
#| export
def decision_learner(data:TypedDecisions, encoder=ENCODER, train="head", cache="auto", head:str="pair", query:str|None="stream:protein",
                     sequence_key:str="sequence", protein_len:int=1024, crop:str="ends", standardize:bool=True,
                     cache_dtype:str="float32", **kw) -> Learner:
    "A protein `Learner`: ProtST with the protein as a side stream, a head that scores each option against the protein, cached features for frozen towers"
    spec = encoder if isinstance(encoder, EncoderSpec) else protst_spec(encoder, protein_len=protein_len, crop=crop)
    if not any(isinstance(t, Stream) for t in data.tfms):
        s = Stream(sequence_key, "protein"); s._added = True
        data.tfms = [s] + list(data.tfms)
    if cache == "auto": cache = "masks" if train == "head" and not kw.get("head_layers") else None
    return _decision_learner(data, spec, train=train, cache=cache, head=head, query=query, standardize=standardize,
                             cache_dtype=cache_dtype, **kw)

On the fixture, 40 proteins with a location (a choice of four) and a membrane noul. Each location draws half its residues from a pool of five amino acids, so even a random ESM's mean carries the location:

In [ ]:
AA, rng = "ACDEFGHIKLMNPQRSTVWY", np.random.default_rng(0)
def fixture_protein(i):
    c = i % 4
    seq = "M" + "".join(rng.choice(list(AA[5 * c:5 * c + 5] if rng.random() < 0.5 else AA)) for _ in range(rng.integers(30, 90)))
    return {"id": f"p{i}", "state": {"sequence": seq}, "questions": QS, "gold": {"location": list(LOCS)[c], "membrane": c == 2}}

precs = [fixture_protein(i) for i in range(40)]
pdata = TypedDecisions.from_records(precs, valid=0.25, calib=0.25, seed=0)
cdir = tempfile.mkdtemp()
torch.manual_seed(0)
learn = decision_learner(pdata, p, preset="cpu", loss="soft_ce", cache_dir=cdir, calibrate_after_fit=False)
calls = []
hook = p.esm.register_forward_pre_hook(lambda m, a, kw: calls.append(tuple(kw["input_ids"].shape)), with_kwargs=True)
ctrain = learn.dataset("train")
hook.remove()
test_eq((len(ctrain), sum(n for n, _ in calls)), (2 * len(pdata.cases["train"]), len(pdata.cases["train"])))   # 2 rows per protein, 1 ESM pass
test_eq([L for _, L in calls], sorted(L for _, L in calls))                                                  # the proteins in order of length
test_eq((ctrain[0]["query"].shape, ctrain[0]["anchors"].shape[1]), ((16,), 32))
learn

In [ ]:
learn.fit(8, lr=3e-3)
learn.calibrate(min_rows=4)
m = learn.validate()
test_eq(m["accuracy"] > 0.5, True)
{k: round(m[k], 3) for k in ("accuracy", "brier", "n")}

The learner answers like any sysone model, and its export loads back as a `Decider`. A folder checkpoint, like the fixture, is bundled into the export. A Hub checkpoint whose towers didn't train is referenced by id and source, and the loader rebuilds it through the registered `protst` source. `bundle=False` takes that path here:

In [ ]:
r = precs[1]
before = learn.predict(r["state"], r["questions"])
for bundle in (True, False):
    out = learn.export(Path(tempfile.mkdtemp()) / "protein-jev", bundle=bundle)
    jev = Decider.load(out, device="cpu")
    test_eq(jev.predict(r["state"], r["questions"]), before)
    test_eq(json.loads((out / "sysone.json").read_text())["encoder"]["weights"], "encoder" if bundle else "hub")
test_eq(sorted(q.name for q in (out / "streams").iterdir()), ["protein"])
before

An act head needs examples of questions it should not answer. Here a third of the proteins also get a `shortlist` question whose options leave out their true location, with the gold `{"answerable": False}`. The rest get a shortlist that includes it. `act=True` adds the head, the learner trains it beside the option loss, and `calibrate` sets the threshold at which it acts:

In [ ]:
def with_shortlist(r, i):
    gold, others = r["gold"]["location"], [k for k in LOCS if k != r["gold"]["location"]]
    keep = others[:3] if i % 3 == 0 else [gold] + others[:2]
    q = {"type": "choice", "instructions": "Which of these compartments holds the protein?", "criteria": {k: LOCS[k] for k in sorted(keep)}}
    return r | {"questions": r["questions"] | {"shortlist": q},
                "gold": r["gold"] | {"shortlist": {"answerable": False} if i % 3 == 0 else gold}}

arecs = [with_shortlist(r, i) for i, r in enumerate(precs)]
adata = TypedDecisions.from_records(arecs, valid=0.25, calib=0.25, seed=0)
torch.manual_seed(0)
la = decision_learner(adata, p, preset="cpu", loss="soft_ce", act=True, cache_dir=cdir, calibrate_after_fit=False)
la.fit(8, lr=3e-3)
la.calibrate(min_rows=4, act_accuracy=0.6)
a = la.predict(arecs[0]["state"], arecs[0]["questions"])
test_eq((sorted(a["shortlist"]["action"]), la.model.head.act), (["act", "act_probability"], True))
test_eq("first" in la.dataset("train").column_names, True)
a["shortlist"]

::: {.callout-note title="Finding: peft needs a base-model prefix on a stream encoder"}
peft treats any model with a `config.model_type` as a transformers model. On transformers 5 it then walks the model's sub-models to adapt its targets to renamed modules, and reads the top model's `base_model_prefix` for each nested one. `StreamEncoder` borrows its text tower's config, so peft took it for a transformers model and failed on the missing attribute. It now declares an empty prefix, since its towers sit at its top level.
:::

The LoRA regime puts adapters on the text tower alone, following the family's targets. A fresh `ProtST` is loaded for it, since adapters change the towers they wrap:

In [ ]:
torch.manual_seed(0)
ll = decision_learner(pdata, load_protst(FIX), train="lora", preset="cpu", loss="soft_ce", calibrate_after_fit=False)
trainable = [n for n, q in ll.model.encoder.named_parameters() if q.requires_grad]
test_eq((ll.cache, len(trainable) > 0, all(".text." in n and "lora_" in n for n in trainable)), (None, True, True))
ll.fit(1, lr=1e-3)
test_eq(ll.model.encoder_trained, True)

## 6. On the real checkpoint

The [protein tutorial](tutorials/04_protein_decisions.ipynb) runs the application on ProtST-ESM1b with every output shown, in 13 minutes on an M3 Pro. It trains on 1,200 proteins from DeepLoc's localization set and asks each of 600 test proteins three questions:

| Question | Zero-shot ProtST | Head on the frozen towers | Baseline |
| --- | --- | --- | --- |
| Location, a choice of 10 | 0.418 (ProtST's label texts as options; 0.270 to 0.345 with other wordings) | 0.828 | 0.300, always *nucleus* |
| Membrane-bound, a noul | 0.758 (Swiss-Prot-style option texts) | 0.906 | 0.578, always *soluble* |
| A shortlist of 4, a third without the right answer | 0.604 on the answerable ones | 0.896 | 0.25, a guess |

The head trains in seconds on vectors cached once, and loses to zero-shot only on the three rarest compartments, which have 17 to 27 training examples each. For escalation, at a 90% target accuracy, thresholds on the head's calibrated top probability act on 70% of the location questions and 31% of the shortlists, against 57% and 3% for the act head's p(act).

## 7. What to watch

- **The port has no tokenizers.** `load_protst` falls back to ESM-1b's and PubMedBERT's, the ones ProtST was trained with. A checkpoint trained with others needs them named (`protein_tokenizer=`, `text_tokenizer=`).
- **Proteins over 1,022 residues are cropped.** ESM-1b has 1,026 positions, and the stream crops at the ends by default. A task that turns on the middle of a long protein would need `crop` changed, or windows.
- **Zero-shot scores are relative.** The cosines between a protein and short texts are small, and ProtST's scale of 14.2 makes them logits. The option texts matter as much as the model does: the port's template names (*SUBCELLULAR LOCATION: Nucleus.*) are the texts ProtST saw in training, and a fluent sentence is not. Calibrate on held-out records before you read the probabilities.
- **The towers are shared.** A spec made from a loaded `ProtST` reuses its modules, and so does every learner made from that spec. The head-only regime leaves them untouched, and LoRA or `full` training changes them for everyone holding them.
- **Acting on the act head.** Laya's act head learns from the rows the option head trains on, where the option head is right far more often than on new proteins, so its p(act) crowds near 1. In the tutorial, thresholds on the head's own calibrated confidence acted on more questions at the same target accuracy. `learn.calibrate(act_score="confidence")` switches the rule, and the export carries it.
- **Memory.** The float32 towers take 3.0 GB. ESM's attention grows with the square of the protein's length, so for 1,024-token proteins keep batches small (8 fits well in 18 GB of unified memory).

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()